# Combined Analysis - Social Bias Across Scale in Ten SLMs
Merges all per-model results, recomputes every metric uniformly, and regenerates the figures and statistical tables (BBQ / CrowS-Pairs / StereoSet, 135M -> 9B).

In [ ]:
# === 1. Imports & global plot style =========================================
# Standard scientific-Python stack; styling kept identical across every figure.
import json, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ROOT = pathlib.Path('.')
SAVE = True

In [ ]:
# === 2. Load summary data ===================================================
# Merge the seven per-family summary CSVs into one frame; attach family/colour/marker.
CSV_PATHS = [
    ROOT / 'smollm2 (contains 135M and 1.7b)' / 'summary_smollm2.csv',
    ROOT / 'qwen2.5 (contains 0.5,1.5 and 3B _ 7b in other)' / 'summary_qwen.csv',
    ROOT / 'tinyllama (only tinyllama_1.1b)' / 'summary_tinyllama.csv',
    ROOT / 'gemma2 (only2b, 9b in other)' / 'summary_gemma (1).csv',
    ROOT / 'phi2_2.7b'    / 'summary_phi.csv',
    ROOT / 'qwen7b'       / 'summary_qwen7b.csv',
    ROOT / 'gemma9b'      / 'summary_gemma9b.csv',
]

df = pd.concat([pd.read_csv(p) for p in CSV_PATHS], ignore_index=True)
df = df.sort_values('params_b').reset_index(drop=True)

FAMILY_MAP = {
    'SmolLM2-135M' : 'SmolLM2',
    'SmolLM2-1.7B' : 'SmolLM2',
    'Qwen2.5-0.5B' : 'Qwen2.5',
    'Qwen2.5-1.5B' : 'Qwen2.5',
    'Qwen2.5-3B'   : 'Qwen2.5',
    'Qwen2.5-7B'   : 'Qwen2.5',
    'TinyLlama-1.1B': 'TinyLlama',
    'Gemma2-2B'    : 'Gemma2',
    'Gemma2-9B'    : 'Gemma2',
    'Phi-2'        : 'Phi',
}
FAMILY_COLORS = {
    'SmolLM2'  : '#4C9BE8',
    'Qwen2.5'  : '#E07B39',
    'TinyLlama': '#5BB85D',
    'Gemma2'   : '#9B59B6',
    'Phi'      : '#E74C3C',
}
df['family'] = df['model'].map(FAMILY_MAP)
df['color']  = df['family'].map(FAMILY_COLORS)
df['log_params'] = np.log10(df['params_b'])

print(df[['model','params_b','family','MBI','bbq_norm','crows_norm','stereoset_norm']].to_string(index=False))

In [ ]:
# === 3. MBI vs scale - all ten models =======================================
# Headline plot: Mean Bias Index against log-parameters with a log-linear fit.
fig, ax = plt.subplots(figsize=(9, 5))

slope, intercept, r, p, _ = stats.linregress(df['log_params'], df['MBI'])
x_line = np.linspace(df['log_params'].min() - 0.05, df['log_params'].max() + 0.05, 200)
ax.plot(10**x_line, slope * x_line + intercept,
        color='#888', lw=1.4, ls='--', zorder=1,
        label=f'log-linear fit  (r={r:.2f}, p={p:.3f})')

for _, row in df.iterrows():
    ax.scatter(row['params_b'], row['MBI'],
               color=row['color'], s=110, zorder=3, edgecolors='white', lw=0.7)
    ax.annotate(row['model'], (row['params_b'], row['MBI']),
                textcoords='offset points', xytext=(5, 4),
                fontsize=7.5, color='#333')

handles = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
handles.append(plt.Line2D([0],[0], color='#888', ls='--', lw=1.4,
                           label=f'log-linear fit  (r={r:.2f}, p={p:.3f})'))
ax.legend(handles=handles, fontsize=8, framealpha=0.8)

ax.set_xscale('log')
ax.set_xlabel('Model size (B parameters, log scale)', fontsize=11)
ax.set_ylabel('Mean Bias Index (MBI)', fontsize=11)
ax.set_title('MBI vs Parameter Count — Full Model Range', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:g}B'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.set_ylim(0.14, 0.30)
plt.tight_layout()
if SAVE: fig.savefig('fig1_mbi_vs_scale.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 4. Per-benchmark normalised bias vs scale ==============================
# Same x-axis, one panel per benchmark - this is where BBQ and StereoSet diverge.
BENCH_COLS   = ['bbq_norm', 'crows_norm', 'stereoset_norm']
BENCH_LABELS = ['BBQ', 'CrowS-Pairs', 'StereoSet']
BENCH_COLORS = ['#2980B9', '#E67E22', '#27AE60']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)

for ax, col, label, bcolor in zip(axes, BENCH_COLS, BENCH_LABELS, BENCH_COLORS):
    slope, intercept, r, p, _ = stats.linregress(df['log_params'], df[col])
    x_line = np.linspace(df['log_params'].min()-0.05, df['log_params'].max()+0.05, 200)
    ax.plot(10**x_line, slope*x_line + intercept,
            color='#999', lw=1.3, ls='--', zorder=1)

    for _, row in df.iterrows():
        ax.scatter(row['params_b'], row[col],
                   color=row['color'], s=90, zorder=3, edgecolors='white', lw=0.6)
        short = (row['model'].replace('Qwen2.5-','Q').replace('SmolLM2-','S')
                 .replace('Gemma2-','G').replace('TinyLlama-','TL').replace('Phi-','Phi'))
        ax.annotate(short, (row['params_b'], row[col]),
                    textcoords='offset points', xytext=(4, 3), fontsize=6.5, color='#333')

    ax.set_xscale('log')
    ax.set_title(f'{label}\n(r={r:.2f}, p={p:.3f})', fontsize=11, fontweight='bold', color=bcolor)
    ax.set_xlabel('Params (B)', fontsize=9)
    ax.set_ylabel('Normalised bias index', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:g}B'))

fig.suptitle('Per-Benchmark Normalised Bias vs Scale', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
if SAVE: fig.savefig('fig2_per_bench_vs_scale.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 5. Stacked benchmark contribution ======================================
df_bar = df.sort_values('params_b')
labels  = [f"{r['model']}\n({r['params_b']}B)" for _, r in df_bar.iterrows()]
bbq     = df_bar['bbq_norm'].values
crows   = df_bar['crows_norm'].values
ss      = df_bar['stereoset_norm'].values

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(labels))
h = 0.6

ax.barh(y, bbq,                height=h, color='#2980B9', label='BBQ')
ax.barh(y, crows, left=bbq,    height=h, color='#E67E22', label='CrowS-Pairs')
ax.barh(y, ss,    left=bbq+crows, height=h, color='#27AE60', label='StereoSet')

mbi_vals = df_bar['MBI'].values
ax.scatter(mbi_vals * 3, y, color='black', zorder=5, s=55, marker='D',
           label='MBI x3 (reference)')

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Cumulative normalised bias  (BBQ + CrowS + StereoSet)', fontsize=10)
ax.set_title('Benchmark Contribution to Total Bias — Sorted by Parameter Count',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.axvline(x=0.5, color='#aaa', lw=0.8, ls=':')
plt.tight_layout()
if SAVE: fig.savefig('fig3_stacked_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 6. BBQ accuracy - ambiguous vs disambiguated ===========================
# Capability proxy: disambiguated accuracy rises with scale even where bias does not.
fig, ax = plt.subplots(figsize=(9, 5))

for _, row in df.iterrows():
    ax.scatter(row['params_b'], row['bbq_acc_disambig'],
               color=row['color'], s=100, zorder=3, edgecolors='white', lw=0.7, marker='o')
    ax.scatter(row['params_b'], row['bbq_acc_ambig'],
               color=row['color'], s=100, zorder=3, edgecolors='white', lw=0.7, marker='^')
    ax.plot([row['params_b'], row['params_b']],
            [row['bbq_acc_ambig'], row['bbq_acc_disambig']],
            color=row['color'], lw=0.9, alpha=0.5, zorder=2)
    ax.annotate(row['model'].split('-')[-1],
                (row['params_b'], row['bbq_acc_disambig']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.set_xscale('log')
ax.set_xlabel('Params (B, log scale)', fontsize=11)
ax.set_ylabel('BBQ Accuracy', fontsize=11)
ax.set_title('BBQ Accuracy: Disambiguated (●) vs Ambiguous (▲)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:g}B'))

legend_elems = [
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#555', ms=9, label='Disambiguated'),
    plt.Line2D([0],[0], marker='^', color='w', markerfacecolor='#555', ms=9, label='Ambiguous'),
] + [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=legend_elems, fontsize=8, framealpha=0.8)

plt.tight_layout()
if SAVE: fig.savefig('fig4_bbq_accuracy.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 7. Within-family comparisons ===========================================
# Hold architecture/data fixed; read off the trajectory as a family scales up.
MULTI_FAMILIES = ['Qwen2.5', 'SmolLM2', 'Gemma2']
METRICS = {
    'MBI'             : 'Mean Bias Index',
    'bbq_norm'        : 'BBQ norm. bias',
    'crows_norm'      : 'CrowS norm. bias',
    'stereoset_norm'  : 'StereoSet norm. bias',
    'bbq_acc_disambig': 'BBQ acc. (disambig.)',
}

fig, axes = plt.subplots(len(MULTI_FAMILIES), len(METRICS),
                         figsize=(16, 9), sharex='row')

for row_i, fam in enumerate(MULTI_FAMILIES):
    sub = df[df['family'] == fam].sort_values('params_b')
    fcolor = FAMILY_COLORS[fam]
    x_vals = sub['params_b'].values
    x_lbls = [r['model'].split('-')[-1] for _, r in sub.iterrows()]

    for col_j, (metric, title) in enumerate(METRICS.items()):
        ax = axes[row_i][col_j]
        y_vals = sub[metric].values

        ax.plot(range(len(x_vals)), y_vals, 'o-',
                color=fcolor, lw=2, ms=8, mec='white', mew=0.8)
        for xi, yv in enumerate(y_vals):
            ax.annotate(f'{yv:.3f}', (xi, yv),
                        textcoords='offset points', xytext=(0, 7),
                        fontsize=7, ha='center', color='#333')

        ax.set_xticks(range(len(x_vals)))
        ax.set_xticklabels([f'{l}\n({xv}B)' for l, xv in zip(x_lbls, x_vals)], fontsize=7.5)
        if col_j == 0:
            ax.set_ylabel(fam, fontsize=10, fontweight='bold', color=fcolor)
        if row_i == 0:
            ax.set_title(title, fontsize=9, fontweight='bold')

fig.suptitle('Within-Family Scaling: How Each Metric Changes as Models Grow',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
if SAVE: fig.savefig('fig5_within_family.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 8. Cross-benchmark correlation matrix ==================================
# Pearson + Spearman; the BBQ-StereoSet cell is the negative correlation we report.
BENCH_COLS   = ['bbq_norm', 'crows_norm', 'stereoset_norm']
BENCH_LABELS_LOCAL = ['BBQ', 'CrowS-Pairs', 'StereoSet']
bench_df = df[BENCH_COLS].copy()
bench_df.columns = BENCH_LABELS_LOCAL

corr_p = bench_df.corr(method='pearson')
corr_s = bench_df.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, corr, method in zip(axes, [corr_p, corr_s], ['Pearson r', 'Spearman rho']):
    sns.heatmap(corr, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=-1, vmax=1, center=0, linewidths=0.5, square=True,
                annot_kws={'size': 12})
    ax.set_title(f'{method} correlation', fontsize=11, fontweight='bold')

fig.suptitle('Cross-Benchmark Correlation (10 models)', fontsize=13, fontweight='bold')
plt.tight_layout()
if SAVE: fig.savefig('fig6_corr_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 9. Pairwise benchmark agreement ========================================
pairs = [
    ('bbq_norm', 'crows_norm',      'BBQ',        'CrowS-Pairs'),
    ('bbq_norm', 'stereoset_norm',  'BBQ',        'StereoSet'),
    ('crows_norm','stereoset_norm', 'CrowS-Pairs','StereoSet'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (cx, cy, lx, ly) in zip(axes, pairs):
    for _, row in df.iterrows():
        ax.scatter(row[cx], row[cy], color=row['color'],
                   s=90, zorder=3, edgecolors='white', lw=0.6)
        short = (row['model'].replace('Qwen2.5-','Q').replace('SmolLM2-','S')
                 .replace('Gemma2-','G').replace('TinyLlama-','TL').replace('Phi-','Ph'))
        ax.annotate(short, (row[cx], row[cy]),
                    textcoords='offset points', xytext=(4, 3), fontsize=7)

    r_p, p_p = pearsonr(df[cx], df[cy])
    r_s, p_s = spearmanr(df[cx], df[cy])

    m, b = np.polyfit(df[cx], df[cy], 1)
    xl = np.linspace(df[cx].min()-0.01, df[cx].max()+0.01, 100)
    ax.plot(xl, m*xl+b, color='#888', lw=1.2, ls='--')

    ax.set_xlabel(lx + ' norm. bias', fontsize=9)
    ax.set_ylabel(ly + ' norm. bias', fontsize=9)
    ax.set_title(f'{lx} vs {ly}\nr={r_p:.2f} (Pearson), rho={r_s:.2f} (Spearman)',
                 fontsize=9, fontweight='bold')

fig.suptitle('Pairwise Benchmark Agreement — Model Rankings', fontsize=13, fontweight='bold')
plt.tight_layout()
if SAVE: fig.savefig('fig7_pairwise_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 10. Per-model radar profiles ===========================================
RADAR_METRICS = ['bbq_norm', 'crows_norm', 'stereoset_norm', 'bbq_acc_disambig']
RADAR_LABELS  = ['BBQ\nbias', 'CrowS\nbias', 'StereoSet\nbias', 'BBQ\nacc.(dis.)']

norm_df = df[RADAR_METRICS].copy()
for col in RADAR_METRICS:
    col_min = norm_df[col].min()
    col_max = norm_df[col].max()
    norm_df[col] = (norm_df[col] - col_min) / (col_max - col_min + 1e-9)

N = len(RADAR_METRICS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

n_models = len(df)
cols_r = 5
rows_r = (n_models + cols_r - 1) // cols_r
fig, axes = plt.subplots(rows_r, cols_r, figsize=(16, rows_r*3.5 + 0.5),
                          subplot_kw=dict(polar=True))
axes = axes.flatten()

for i, (_, row) in enumerate(df.iterrows()):
    ax = axes[i]
    vals = norm_df.iloc[i].tolist()
    vals += vals[:1]

    ax.plot(angles, vals, color=row['color'], lw=2)
    ax.fill(angles, vals, color=row['color'], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(RADAR_LABELS, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['', '0.5', '', '1.0'], fontsize=6)
    ax.set_ylim(0, 1)
    ax.set_title(f"{row['model']}\n({row['params_b']}B)",
                 fontsize=8, fontweight='bold', color=row['color'], pad=10)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Per-Model Bias Profile (Radar) — axes normalised 0-1 across all models',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
if SAVE: fig.savefig('fig8_radar.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 11. Per-category BBQ abstention (from JSONL) ===========================
# Re-reads raw predictions; abstention = model chose 'unknown' in an ambiguous context.
BBQ_JSONL = {}
for path in sorted(ROOT.rglob('*__bbq.jsonl')):
    model_name = path.stem.replace('__bbq', '')
    BBQ_JSONL[model_name] = path
print('Found BBQ JSONLs for:', list(BBQ_JSONL.keys()))

BBQ_CATS = [
    'Age','Disability_status','Gender_identity','Nationality',
    'Physical_appearance','Race_ethnicity','Race_x_SES',
    'Race_x_gender','Religion','SES','Sexual_orientation'
]

def get_category(uid):
    for cat in BBQ_CATS:
        if uid.startswith(cat):
            return cat
    return 'Unknown'

rows_list = []
for model_name, path in BBQ_JSONL.items():
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            rec['model'] = model_name
            rec['category'] = get_category(rec['uid'])
            rows_list.append(rec)

bbq_raw = pd.DataFrame(rows_list)

def item_index(uid, cat):
    try:
        return int(uid.replace(cat+'_', ''))
    except:
        return -1

bbq_raw['item_idx'] = bbq_raw.apply(lambda r: item_index(r['uid'], r['category']), axis=1)
ambig = bbq_raw[bbq_raw['item_idx'] < 1000].copy()
ambig['abstained'] = ambig['pred'] == 2

cat_model = ambig.groupby(['model','category'])['abstained'].mean().reset_index()
cat_model.columns = ['model','category','abstention_rate']
cat_model = cat_model.merge(df[['model','params_b']], on='model', how='left')

model_order = df.sort_values('params_b')['model'].tolist()
pivot = cat_model.pivot(index='model', columns='category', values='abstention_rate')
pivot = pivot.reindex([m for m in model_order if m in pivot.index])

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd_r', vmin=0, vmax=0.15,
            annot=True, fmt='.2f', linewidths=0.3, linecolor='#eee',
            annot_kws={'size': 7.5})
ax.set_title(
    'BBQ Ambiguous-Context Abstention Rate by Category\n'
    '(fraction choosing "unknown"; higher = less biased)',
    fontsize=12, fontweight='bold')
ax.set_xlabel('BBQ Category', fontsize=10)
ax.set_ylabel('Model (sorted by params)', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
plt.tight_layout()
if SAVE: fig.savefig('fig9_bbq_category_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 12. CrowS-Pairs bias-type breakdown ====================================
CROWS_JSONL = {}
for path in sorted(ROOT.rglob('*__crows.jsonl')):
    model_name = path.stem.replace('__crows', '')
    CROWS_JSONL[model_name] = path

c_rows = []
for model_name, path in CROWS_JSONL.items():
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            rec['model'] = model_name
            rec['stereo_preferred'] = rec['score_more'] > rec['score_less']
            c_rows.append(rec)

crows_raw = pd.DataFrame(c_rows)
crows_raw = crows_raw.merge(df[['model','params_b']], on='model', how='left')

bias_type_model = (crows_raw.groupby(['model','bias_type'])['stereo_preferred']
                   .mean().reset_index())
bias_type_model.columns = ['model','bias_type','stereo_pct']
bias_type_model = bias_type_model.merge(df[['model','params_b']], on='model', how='left')

pivot_c = bias_type_model.pivot(index='model', columns='bias_type', values='stereo_pct')
pivot_c = pivot_c.reindex([m for m in model_order if m in pivot_c.index])

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot_c, ax=ax, cmap='RdYlGn_r', vmin=0.3, vmax=0.8, center=0.5,
            annot=True, fmt='.2f', linewidths=0.3, linecolor='#eee',
            annot_kws={'size': 8})
ax.set_title(
    'CrowS-Pairs Stereotype Preference Rate by Bias Type\n(>0.5 = pro-stereotype; red = more biased)',
    fontsize=12, fontweight='bold')
ax.set_xlabel('Bias Type', fontsize=10)
ax.set_ylabel('Model (sorted by params)', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
plt.tight_layout()
if SAVE: fig.savefig('fig10_crows_biastype_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 13. StereoSet bias-domain breakdown ====================================
# LMS filter applied before SS so fluency is not folded into the bias term.
SS_JSONL = {}
for path in sorted(ROOT.rglob('*__stereoset.jsonl')):
    model_name = path.stem.replace('__stereoset', '')
    SS_JSONL[model_name] = path

ss_rows = []
for model_name, path in SS_JSONL.items():
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            rec['model'] = model_name
            rec['lm_correct'] = max(rec['score_stereo'], rec['score_anti']) > rec['score_unrelated']
            rec['picks_stereo'] = rec['score_stereo'] > rec['score_anti']
            ss_rows.append(rec)

ss_raw = pd.DataFrame(ss_rows)

def ss_domain_stats(group):
    valid = group[group['lm_correct']]
    ss_val = valid['picks_stereo'].mean() * 100 if len(valid) else float('nan')
    lms_val = group['lm_correct'].mean() * 100
    return pd.Series({'SS': ss_val, 'LMS': lms_val, 'n': len(group)})

ss_domain = ss_raw.groupby(['model','bias_type']).apply(ss_domain_stats).reset_index()
ss_domain = ss_domain.merge(df[['model','params_b']], on='model', how='left')

pivot_ss = ss_domain.pivot(index='model', columns='bias_type', values='SS')
pivot_ss = pivot_ss.reindex([m for m in model_order if m in pivot_ss.index])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_ss, ax=ax, cmap='RdYlGn_r', vmin=40, vmax=80, center=50,
            annot=True, fmt='.1f', linewidths=0.3, linecolor='#eee',
            annot_kws={'size': 9})
ax.set_title(
    'StereoSet Stereotype Score (SS) by Bias Domain\n(>50 = pro-stereotype; red = more biased)',
    fontsize=12, fontweight='bold')
ax.set_xlabel('Bias Domain', fontsize=10)
ax.set_ylabel('Model (sorted by params)', fontsize=10)
plt.tight_layout()
if SAVE: fig.savefig('fig11_stereoset_domain_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# === 14. Statistical summary ================================================
# Correlations of each metric with log-params - the formal H1-vs-H2 test.
results = {}
for col, name in [('MBI','MBI'), ('bbq_norm','BBQ'), ('crows_norm','CrowS'), ('stereoset_norm','StereoSet')]:
    sl, intc, r_p, p_p, _ = stats.linregress(df['log_params'], df[col])
    r_s, p_s = spearmanr(df['params_b'], df[col])
    interp = 'n.s. (weak)' if p_p > 0.1 else ('larger=MORE biased' if sl > 0 else 'larger=LESS biased')
    results[name] = {
        'Pearson r': round(r_p, 3),
        'Pearson p': round(p_p, 4),
        'Spearman rho': round(r_s, 3),
        'Spearman p': round(p_s, 4),
        'Slope/log10B': round(sl, 4),
        'Interpretation': interp,
    }

stats_df = pd.DataFrame(results).T
print('=== Correlation with log(params) ===')
display(stats_df)

print(f"\nMin MBI : {df['MBI'].min():.4f}  ({df.loc[df['MBI'].idxmin(),'model']})")
print(f"Max MBI : {df['MBI'].max():.4f}  ({df.loc[df['MBI'].idxmax(),'model']})")
print(f"Range   : {df['MBI'].max()-df['MBI'].min():.4f}")
print(f"Mean    : {df['MBI'].mean():.4f}")
print(f"Std     : {df['MBI'].std():.4f}")

In [ ]:
# === 15. Export master CSV ==================================================
out = df.drop(columns=['color','log_params']).sort_values('params_b')
out.to_csv('combined_summary.csv', index=False)
print('Saved combined_summary.csv')
display(out)